# SPAI Inference Results

Objective: present saved SPAI detector outputs for NTIRE and Z-Image-Turbo without constructing the model or dataloaders by default.

Artifacts preserved by this notebook:
- `data/silver/spai/ntire_scores.csv`
- `data/silver/spai/z_image_turbo_scores.csv`
- `data/silver/spai/ntire_metrics.json`
- `data/silver/spai/experiment_config.json`
- `data/silver/spai/plots/`


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import SVG, display
from sklearn.metrics import precision_recall_curve, roc_curve

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent.parent

OUTPUT_DIR = REPO_ROOT / "data" / "silver" / "spai"
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

NTIRE_SCORES = OUTPUT_DIR / "ntire_scores.csv"
ZIT_SCORES = OUTPUT_DIR / "z_image_turbo_scores.csv"
NTIRE_METRICS = OUTPUT_DIR / "ntire_metrics.json"
EXPERIMENT_CONFIG = OUTPUT_DIR / "experiment_config.json"

for path in [NTIRE_SCORES, ZIT_SCORES, NTIRE_METRICS, EXPERIMENT_CONFIG]:
    if not path.exists():
        raise FileNotFoundError(path)

print(f"Repo root: {REPO_ROOT}")
print(f"SPAI artifacts: {OUTPUT_DIR}")

In [ ]:
results_ntire = pd.read_csv(NTIRE_SCORES)
results_zit = pd.read_csv(ZIT_SCORES)
with NTIRE_METRICS.open() as handle:
    metrics_ntire = json.load(handle)
with EXPERIMENT_CONFIG.open() as handle:
    experiment_config = json.load(handle)

summary_ntire = pd.DataFrame(
    [
        {
            "dataset": "NTIRE",
            "images": len(results_ntire),
            "accuracy": metrics_ntire.get("accuracy"),
            "average_precision": metrics_ntire.get("average_precision"),
            "auc_roc": metrics_ntire.get("auc_roc"),
            "f1": metrics_ntire.get("f1"),
            "precision": metrics_ntire.get("precision"),
            "recall": metrics_ntire.get("recall"),
        }
    ]
)
summary_zit = pd.DataFrame(
    [
        {
            "dataset": "Z-Image-Turbo",
            "images": len(results_zit),
            "mean_score": results_zit["score"].mean(),
            "std_score": results_zit["score"].std(),
            "median_score": results_zit["score"].median(),
            "score_gt_0_5_rate": (results_zit["score"] > 0.5).mean(),
        }
    ]
)

display(summary_ntire.style.format(precision=4))
display(summary_zit.style.format(precision=4))

In [ ]:
y_true = results_ntire["label"].to_numpy()
y_score = results_ntire["score"].to_numpy()

fig, ax = plt.subplots(figsize=(6, 4))
for label, color, name in [(0, "#276FBF", "NTIRE real"), (1, "#C84630", "NTIRE fake")]:
    subset = results_ntire.loc[results_ntire["label"] == label, "score"]
    ax.hist(subset, bins=40, alpha=0.55, label=name, color=color, density=True)
ax.hist(
    results_zit["score"],
    bins=25,
    alpha=0.55,
    label="Z-Image-Turbo fake",
    color="#2A9D8F",
    density=True,
)
ax.axvline(0.5, color="black", linestyle="--", linewidth=0.9)
ax.set_xlabel("Fake probability")
ax.set_ylabel("Density")
ax.set_title("SPAI score distribution")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "score_histogram.svg", format="svg")
plt.close(fig)

fpr, tpr, _ = roc_curve(y_true, y_score)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(
    fpr, tpr, color="#C84630", linewidth=2, label=f"AUC={metrics_ntire['auc_roc']:.4f}"
)
ax.plot([0, 1], [0, 1], "k--", linewidth=0.9)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("SPAI ROC on NTIRE")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ntire_roc_curve.svg", format="svg")
plt.close(fig)

precision, recall, _ = precision_recall_curve(y_true, y_score)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(recall, precision, color="#C84630", linewidth=2)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"SPAI PR on NTIRE (AP={metrics_ntire['average_precision']:.4f})")
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ntire_pr_curve.svg", format="svg")
plt.close(fig)

print(f"Saved SVG plots to {PLOTS_DIR}")
display(SVG(filename=PLOTS_DIR / "score_histogram.svg"))

In [ ]:
display(SVG(filename=PLOTS_DIR / "ntire_roc_curve.svg"))
display(SVG(filename=PLOTS_DIR / "ntire_pr_curve.svg"))

In [ ]:
RUN_INFERENCE = False

if RUN_INFERENCE:
    # Long-running path: prefer the project CLI/background process for full regeneration.
    # Example:
    # uv run python -m diffguard.cli.run_spai_inference
    raise RuntimeError(
        "SPAI inference is intentionally not run inside the notebook. "
        "Use the CLI/background job and keep this notebook for presentation."
    )
else:
    print(
        "Inference cell skipped. Use the SPAI CLI/background job only when regenerating scores."
    )

## Notes

SPAI uses a frequency-restoration-oriented detector and performs better than AIDE on NTIRE in the saved run, while AIDE is more sensitive on Z-Image-Turbo. These outputs feed the gold-layer comparison notebook.
